# 🗂 Bases de datos vectoriales

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 4 — 45 minutos**

---

## Objetivo

Almacenar embeddings de texto en ChromaDB y realizar búsqueda semántica — el corazón de todo sistema RAG.

## Al terminar este bloque vas a poder:

1. Crear colecciones en ChromaDB y agregar documentos con metadata.
2. Buscar por similitud semántica y compararlo con búsqueda por palabras exactas.
3. Usar un modelo multilingüe para mejorar resultados en español argentino.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Base de datos vectorial** | Motor especializado en guardar y consultar embeddings con baja latencia. |
| **ChromaDB** | Base vectorial open-source embebida en Python — sin servidor externo. |
| **Similitud coseno** | Mide la cercanía entre dos vectores: 1 = idénticos, 0 = sin relación. |
| **Colección** | Grupo de documentos relacionados en ChromaDB, como una tabla en SQL. |
| **CRUD vectorial** | `add` / `get` / `update` / `delete` / `query` — las operaciones básicas. |

In [ ]:
%pip install -qq chromadb sentence-transformers

In [ ]:
# Verificamos que PyTorch está instalado (necesario para sentence-transformers)
import torch
print(f"PyTorch versión: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

## ¿Para qué sirve una base de datos vectorial?

### Analogía

Una base SQL busca por coincidencia exacta: si preguntás por 'asado', solo te devuelve registros que tengan la palabra 'asado'. Una base vectorial busca por significado: 'carne a las brasas', 'parrilla', 'bife' son conceptualmente cercanos a 'asado' aunque no compartan ninguna letra.

### Dónde vive esto en el mundo real

ChromaDB, Pinecone y Weaviate son la infraestructura invisible de los asistentes de documentos de Notion, Slack y GitHub Copilot. Cada vez que un chatbot empresarial responde con información de un manual específico, hay una base vectorial detrás recuperando los fragmentos más relevantes. En el próximo notebook vas a conectar ChromaDB con Gemini para construir exactamente eso.

### ✎ Para pensar

- ¿Por qué ChromaDB usa similitud coseno en lugar de distancia euclidiana para comparar embeddings?
- Si agregás un documento en inglés a una colección con textos en español, ¿el modelo base (`all-MiniLM`) lo va a comparar bien con consultas en español?

In [2]:
import chromadb

# Creamos un cliente de ChromaDB en memoria
# Nota: Los datos se pierden al cerrar el notebook
# Para persistencia, usar: chromadb.PersistentClient(path="./chroma_db")
client = chromadb.Client()

print("ChromaDB inicializado correctamente")

ChromaDB inicializado correctamente


## Demo: base de reviews de restaurantes porteños

Vamos a construir un recomendador semántico con reviews reales. El objetivo es que una consulta como 'algo barato para morfar' encuentre restaurantes relevantes aunque no usen esas palabras exactas.

In [3]:
collection = client.get_or_create_collection(
    name="reviews_restaurantes"
)
print("Colección 'reviews_restaurantes' lista")

Colección 'reviews_restaurantes' lista


In [4]:
# Reviews iniciales - variedad de restaurantes en CABA
reviews_iniciales = [
    "Fui a cenar pasta y la verdad que espectacular. Los ñoquis con salsa fileto increíbles. Precio razonable, como 8 lucas por persona. Ambiente tranquilo, perfecto para ir en pareja.",

    "La mejor pizza que comí en mi vida, no jodo. Masa finita, crocante, mucha muzzarella. Eso sí, siempre está lleno y hay que esperar. Vale la pena. Estilo porteño posta.",

    "Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re elaborados. Caro pero para una ocasión especial está bueno. Tiene buen vino también.",

    "Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las papas fritas caseras. Atención familiar, te hacen sentir como en tu casa. Precios normales.",

    "Sushi delivery que pedimos seguido. Fresco, bien armado, llega rápido. No es el mejor que probé pero para el precio está más que bien. El combo para dos es suficiente."
]

# Metadatos: información adicional sobre cada review
metadatos_iniciales = [
    {"barrio": "San Telmo", "tipo": "italiana", "precio": "medio"},
    {"barrio": "Palermo", "tipo": "pizzeria", "precio": "medio"},
    {"barrio": "Palermo", "tipo": "gourmet", "precio": "alto"},
    {"barrio": "Villa Urquiza", "tipo": "parrilla", "precio": "medio"},
    {"barrio": "delivery", "tipo": "sushi", "precio": "medio"}
]

# IDs únicos para cada documento
ids_iniciales = ["review1", "review2", "review3", "review4", "review5"]

### Estructura de una colección

Cada `add()` necesita tres listas de igual longitud:

| Parámetro | Contenido |
|---|---|
| `documents` | Los textos en sí |
| `metadatas` | Diccionarios con datos extra (barrio, tipo, precio) |
| `ids` | Identificadores únicos para cada documento |

In [5]:
# Agregamos las reviews a la colección
collection.add(
    documents=reviews_iniciales,
    metadatas=metadatos_iniciales,
    ids=ids_iniciales
)

print(f"Se agregaron {len(reviews_iniciales)} reviews a la base de datos")

Se agregaron 5 reviews a la base de datos


In [6]:
# Verificamos cuántos documentos tenemos
collection.count()

5

In [7]:
# Obtenemos todos los documentos para verificar
todos = collection.get()
print("Documentos en la colección:")
for i, doc in enumerate(todos['documents'], 1):
    print(f"\n{i}. {doc[:80]}...")

Documentos en la colección:

1. Fui a cenar pasta y la verdad que espectacular. Los ñoquis con salsa fileto incr...

2. La mejor pizza que comí en mi vida, no jodo. Masa finita, crocante, mucha muzzar...

3. Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re e...

4. Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las ...

5. Sushi delivery que pedimos seguido. Fresco, bien armado, llega rápido. No es el ...


## Búsqueda semántica vs búsqueda por palabras

Vas a comparar los dos tipos de búsqueda con los mismos datos para ver concretamente la diferencia.

In [8]:
# Búsqueda 1: Quiero comer algo italiano
consulta = "Busco un lugar para comer buena comida italiana, tipo ravioles o ñoquis"

resultados = collection.query(
    query_texts=[consulta],
    n_results=3  # Los 3 más similares
)

print(f"CONSULTA: {consulta}")
print("\nREVIEWS MÁS SIMILARES:")
print("=" * 80)
for i, (doc, metadata) in enumerate(zip(resultados['documents'][0], resultados['metadatas'][0]), 1):
    print(f"\n{i}. {doc}")
    print(f"   Barrio: {metadata['barrio']}, Tipo: {metadata['tipo']}, Precio: {metadata['precio']}")

CONSULTA: Busco un lugar para comer buena comida italiana, tipo ravioles o ñoquis

REVIEWS MÁS SIMILARES:

1. Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re elaborados. Caro pero para una ocasión especial está bueno. Tiene buen vino también.
   Barrio: Palermo, Tipo: gourmet, Precio: alto

2. Fui a cenar pasta y la verdad que espectacular. Los ñoquis con salsa fileto increíbles. Precio razonable, como 8 lucas por persona. Ambiente tranquilo, perfecto para ir en pareja.
   Barrio: San Telmo, Tipo: italiana, Precio: medio

3. La mejor pizza que comí en mi vida, no jodo. Masa finita, crocante, mucha muzzarella. Eso sí, siempre está lleno y hay que esperar. Vale la pena. Estilo porteño posta.
   Barrio: Palermo, Tipo: pizzeria, Precio: medio


In [9]:
# Búsqueda 2: Lugar económico para carne
consulta2 = "Dónde puedo ir a comer buen asado sin gastar mucha plata?"

resultados2 = collection.query(
    query_texts=[consulta2],
    n_results=3
)

print(f"CONSULTA: {consulta2}")
print("\nREVIEWS MÁS SIMILARES:")
print("=" * 80)
for i, (doc, metadata) in enumerate(zip(resultados2['documents'][0], resultados2['metadatas'][0]), 1):
    print(f"\n{i}. {doc}")
    print(f"   Barrio: {metadata['barrio']}, Tipo: {metadata['tipo']}, Precio: {metadata['precio']}")

CONSULTA: Dónde puedo ir a comer buen asado sin gastar mucha plata?

REVIEWS MÁS SIMILARES:

1. Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las papas fritas caseras. Atención familiar, te hacen sentir como en tu casa. Precios normales.
   Barrio: Villa Urquiza, Tipo: parrilla, Precio: medio

2. Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re elaborados. Caro pero para una ocasión especial está bueno. Tiene buen vino también.
   Barrio: Palermo, Tipo: gourmet, Precio: alto

3. La mejor pizza que comí en mi vida, no jodo. Masa finita, crocante, mucha muzzarella. Eso sí, siempre está lleno y hay que esperar. Vale la pena. Estilo porteño posta.
   Barrio: Palermo, Tipo: pizzeria, Precio: medio


In [10]:
# Búsqueda tradicional: buscar la palabra "asado" exacta
print("BÚSQUEDA TRADICIONAL (palabra exacta 'asado'):")
busqueda_tradicional = collection.get(
    where_document={"$contains": "asado"}
)

if len(busqueda_tradicional['documents']) > 0:
    for doc in busqueda_tradicional['documents']:
        print(f"- {doc}")
else:
    print("No se encontraron resultados con la palabra 'asado'")

print("\n" + "="*80)
print("\nBÚSQUEDA SEMÁNTICA (por significado):")
print("Encontró la parrilla aunque no use la palabra 'asado' exacta")

BÚSQUEDA TRADICIONAL (palabra exacta 'asado'):
No se encontraron resultados con la palabra 'asado'


BÚSQUEDA SEMÁNTICA (por significado):
Encontró la parrilla aunque no use la palabra 'asado' exacta


### ✎ Para pensar

- La búsqueda encontró 'ñoquis' ante la consulta 'comida italiana'. ¿Qué tiene que saber el modelo de embeddings para hacer esa conexión?
- Probá la consulta 'lugar barato para morfar' con el modelo base y el multilingüe. ¿Cuál da mejor resultado? ¿Por qué?

In [11]:
# Búsqueda 3: Lugar barato para morfar. Modelo base, por defecto, no "entendería" la palabra morfar
consulta3 = "Lugar barato para morfar"

resultados3 = collection.query(
    query_texts=[consulta3],
    n_results=3
)

print(f"CONSULTA: {consulta3}")
print("\nREVIEWS MÁS SIMILARES:")
print("=" * 80)
for i, (doc, metadata) in enumerate(zip(resultados2['documents'][0], resultados2['metadatas'][0]), 1):
    print(f"\n{i}. {doc}")
    print(f"   Barrio: {metadata['barrio']}, Tipo: {metadata['tipo']}, Precio: {metadata['precio']}")#

CONSULTA: Lugar barato para morfar

REVIEWS MÁS SIMILARES:

1. Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las papas fritas caseras. Atención familiar, te hacen sentir como en tu casa. Precios normales.
   Barrio: Villa Urquiza, Tipo: parrilla, Precio: medio

2. Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re elaborados. Caro pero para una ocasión especial está bueno. Tiene buen vino también.
   Barrio: Palermo, Tipo: gourmet, Precio: alto

3. La mejor pizza que comí en mi vida, no jodo. Masa finita, crocante, mucha muzzarella. Eso sí, siempre está lleno y hay que esperar. Vale la pena. Estilo porteño posta.
   Barrio: Palermo, Tipo: pizzeria, Precio: medio


In [15]:
# Nuevas reviews que llegan
nuevas_reviews = [
    "Bar de tragos en Palermo re copado. Buenos cócteles, música en vivo los fines de semana. Se llena bastante después de las 11. Tiene terraza.",

    "Cafetería de especialidad, café re rico, tienen opciones veganas. Ambiente tranquilo para trabajar con la compu. WiFi gratis y enchufes.",

    "Bodegón español tradicional. Las croquetas y la tortilla de papa son de otro planeta. Vino de la casa riquísimo. Porteño viejo, 100 años de historia.",

    "Hamburguesería gourmet. Las papas con cheddar y bacon son adictivas. Burgers de 200gr, jugosas. Podes armar la tuya. Delivery hasta las 2am.",

    "Cantina familiar muy casera. La comida es simple pero rica, como la que hace tu nona. Milanesas gigantes. Super económico, 6 lucas comes re bien."
]

nuevos_metadatos = [
    {"barrio": "Palermo", "tipo": "bar", "precio": "medio-alto"},
    {"barrio": "Colegiales", "tipo": "cafeteria", "precio": "medio"},
    {"barrio": "Montserrat", "tipo": "española", "precio": "medio"},
    {"barrio": "Belgrano", "tipo": "hamburguesas", "precio": "medio"},
    {"barrio": "Boedo", "tipo": "casera", "precio": "bajo"}
]

In [16]:
# Función helper para agregar reviews de forma organizada
def agregar_reviews(collection, reviews, metadatos, prefijo_id="review"):
    """
    Agrega nuevas reviews a una colección existente.

    Parámetros:
    -----------
    collection : chromadb.Collection
        La colección donde agregar los documentos
    reviews : list
        Lista de textos de reviews
    metadatos : list
        Lista de diccionarios con metadatos
    prefijo_id : str
        Prefijo para generar IDs únicos
    """
    # Obtenemos cuántos documentos ya hay para generar IDs únicos
    count_actual = collection.count()

    # Generamos IDs únicos
    nuevos_ids = [f"{prefijo_id}{count_actual + i + 1}" for i in range(len(reviews))]

    # Agregamos a la colección
    collection.add(
        documents=reviews,
        metadatas=metadatos,
        ids=nuevos_ids
    )

    print(f"Se agregaron {len(reviews)} nuevas reviews")
    print(f"Total de documentos en la colección: {collection.count()}")
    return nuevos_ids

In [17]:
# Agregamos las nuevas reviews
ids_nuevos = agregar_reviews(collection, nuevas_reviews, nuevos_metadatos)

Se agregaron 5 nuevas reviews
Total de documentos en la colección: 10


## Mejorando con embeddings multilingüe

El modelo por defecto (`all-MiniLM-L6-v2`) fue entrenado mayormente en inglés. Para modismos como 're copado', 'posta', 'morfar', necesitamos `multilingual-e5-large`.

**Trade-off:** más preciso para español, pero más lento y ocupa más memoria.

In [18]:
!uv pip install --upgrade "transformers>=4.40.0,<5.0.0" "sentence-transformers>=3.0.0"
print("✓ ¡Listo! Actualizado con uv en un par de segundos.")

✓ ¡Listo! Actualizado con uv en un par de segundos.


Resolved 31 packages in 302ms
Checked 31 packages in 3ms


In [2]:
!uv pip install ipykernel

Resolved 29 packages in 657ms
Installed 23 packages in 11.65s
 + asttokens==3.0.1
 + comm==0.2.3
 + debugpy==1.8.21
 + decorator==5.3.1
 + executing==2.2.1
 + ipykernel==7.3.0
 + ipython==9.14.1
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 + jupyter-client==8.9.1
 + jupyter-core==5.9.1
 + matplotlib-inline==0.2.2
 + nest-asyncio2==1.7.2
 + parso==0.8.7
 + platformdirs==4.10.0
 + prompt-toolkit==3.0.52
 + psutil==7.2.2
 + pure-eval==0.2.3
 + pyzmq==27.1.0
 + stack-data==0.6.3
 + tornado==6.5.7
 + traitlets==5.15.1
 + wcwidth==0.8.1


In [20]:
from chromadb.utils import embedding_functions

# Configuramos el modelo multilenguaje
# Esto descarga el modelo la primera vez (puede tardar unos minutos)
modelo_multilenguaje = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/multilingual-e5-large"
)

print("Modelo multilenguaje cargado")
print("Este modelo entiende español argentino mucho mejor")


c:\Users\iblis\Desktop\NLP\sanches-sabrina-pln-1c-2026\012\001 - TEO\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\iblis\.cache\huggingface\hub\models--intfloat--multilingual-e5-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Modelo multilenguaje cargado
Este modelo entiende español argentino mucho mejor


In [32]:
# Creamos una nueva colección con el modelo mejorado
collection_mejorada = client.get_or_create_collection(
    name="reviews_restaurantes_multilenguaje",
    embedding_function=modelo_multilenguaje,
    metadata={"hnsw:space": "cosine"}  # Método de comparación de vectores
)

print("Colección con embeddings multilenguaje creada")

Colección con embeddings multilenguaje creada


In [33]:
# Verificamos las colecciones disponibles
print("Colecciones en la base de datos:")
for col in client.list_collections():
    print(f"  - {col.name}")

Colecciones en la base de datos:
  - reviews_restaurantes_multilenguaje
  - reviews_multilenguaje
  - reviews_restaurantes


In [34]:
# Agregamos TODAS las reviews (iniciales + nuevas) a la colección mejorada
todas_las_reviews = reviews_iniciales + nuevas_reviews
todos_los_metadatos = metadatos_iniciales + nuevos_metadatos
todos_los_ids = ids_iniciales + ids_nuevos

collection_mejorada.add(
    documents=todas_las_reviews,
    metadatas=todos_los_metadatos,
    ids=todos_los_ids
)

print(f"Se agregaron {len(todas_las_reviews)} reviews a la colección mejorada")

Se agregaron 10 reviews a la colección mejorada


In [35]:
# Consulta con modismos argentinos
consulta_argentina = "Busco un lugar re copado para morfar algo barato y llenadero"

print("="*80)
print(f"CONSULTA: {consulta_argentina}")
print("="*80)

# Búsqueda con modelo base
print("\n1. MODELO BASE (all-MiniLM-L6-v2):")
print("-"*80)
resultado_base = collection.query(
    query_texts=[consulta_argentina],
    n_results=3
)
for i, doc in enumerate(resultado_base['documents'][0], 1):
    print(f"\n{i}. {doc[:100]}...")

# Búsqueda con modelo multilenguaje
print("\n" + "="*80)
print("\n2. MODELO MULTILENGUAJE (multilingual-e5-large):")
print("-"*80)
resultado_mejorado = collection_mejorada.query(
    query_texts=[consulta_argentina],
    n_results=3
)
for i, (doc, metadata) in enumerate(zip(resultado_mejorado['documents'][0], resultado_mejorado['metadatas'][0]), 1):
    print(f"\n{i}. {doc[:100]}...")
    print(f"   Tipo: {metadata['tipo']}, Precio: {metadata['precio']}")

CONSULTA: Busco un lugar re copado para morfar algo barato y llenadero

1. MODELO BASE (all-MiniLM-L6-v2):
--------------------------------------------------------------------------------

1. Bar de tragos en Palermo re copado. Buenos cócteles, música en vivo los fines de semana. Se llena ba...

2. Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re elaborados. Caro pero...

3. Cantina familiar muy casera. La comida es simple pero rica, como la que hace tu nona. Milanesas giga...


2. MODELO MULTILENGUAJE (multilingual-e5-large):
--------------------------------------------------------------------------------

1. Cafetería de especialidad, café re rico, tienen opciones veganas. Ambiente tranquilo para trabajar c...
   Tipo: cafeteria, Precio: medio

2. Bar de tragos en Palermo re copado. Buenos cócteles, música en vivo los fines de semana. Se llena ba...
   Tipo: bar, Precio: medio-alto

3. Parrilla clásica de barrio. El bife de chorizo una locura, tierno y j

### 🐛 Laboratorio de Rotura

El código de abajo *parece* correcto. Antes de ejecutarlo, predecí: ¿qué va a pasar si intentás crear una colección con un nombre que ya existe?

```python
colision = client.create_collection(name="reviews_restaurantes")
```

Ejecutá la celda siguiente.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Cómo deberías manejar esto en un sistema real que se reinicia?

No mires la explicación todavía.

In [36]:
# ── Momento 1 — La colisión de nombres ──
try:
    colision = client.create_collection(name="reviews_restaurantes")
    print("Se creó... pero ¿cómo puede haber dos colecciones con el mismo nombre?")
except Exception as e:
    print(f"✗ Error: {e}")
    print()
    print("ChromaDB no permite dos colecciones con el mismo nombre en el mismo cliente.")

# ── Momento 2 — La solución idiomática ──
# get_or_create_collection evita el error: si existe la devuelve, si no la crea
collection_segura = client.get_or_create_collection(name="reviews_restaurantes")
print(f"\n✓ get_or_create_collection: {collection_segura.count()} documentos ya cargados")
print("   Usá get_or_create_collection en producción para hacer el sistema resistente a reinicios.")

✗ Error: Collection [reviews_restaurantes] already exists

ChromaDB no permite dos colecciones con el mismo nombre en el mismo cliente.

✓ get_or_create_collection: 10 documentos ya cargados
   Usá get_or_create_collection en producción para hacer el sistema resistente a reinicios.


### 🐛 Laboratorio de Rotura

El código de abajo *parece* correcto. Antes de ejecutarlo, predecí: ¿qué va a pasar si intentás crear una colección con un nombre que ya existe?

```python
colision = client.create_collection(name="reviews_restaurantes")
```

Ejecutá la celda siguiente.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Cómo deberías manejar esto en un sistema real que se reinicia?

No mires la explicación todavía.

In [37]:
# ── Momento 1 — La colisión de nombres ──
try:
    colision = client.create_collection(name="reviews_restaurantes")
    print("Se creó... pero ¿cómo puede haber dos colecciones con el mismo nombre?")
except Exception as e:
    print(f"✗ Error: {e}")
    print()
    print("ChromaDB no permite dos colecciones con el mismo nombre en el mismo cliente.")

# ── Momento 2 — La solución idiomática ──
# get_or_create_collection evita el error: si existe la devuelve, si no la crea
collection_segura = client.get_or_create_collection(name="reviews_restaurantes")
print(f"\n✓ get_or_create_collection: {collection_segura.count()} documentos ya cargados")
print("   Usá get_or_create_collection en producción para hacer el sistema resistente a reinicios.")

✗ Error: Collection [reviews_restaurantes] already exists

ChromaDB no permite dos colecciones con el mismo nombre en el mismo cliente.

✓ get_or_create_collection: 10 documentos ya cargados
   Usá get_or_create_collection en producción para hacer el sistema resistente a reinicios.


In [42]:
# --- Espacio de practica ---
#
# Proba estas busquedas y compara los resultados:
#   1. 'Un lugar romantico para ir con mi pareja'
#   2. 'Algo para comer rapido al mediodia'
#   3. 'Comida como la que hace la abuela'
#
# Para cada una:
#   - Ejecuta con el modelo base (collection)
#   - Ejecuta con el modelo multilingue (collection_mejorada)
#   - Registra si los resultados tienen sentido semantico
#
MI_CONSULTA = 'Un lugar romantico para ir con mi pareja'

mis_resultados = collection.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

print(f'Consulta: {MI_CONSULTA}')
print("="*10, "Modelo BASE", "="*10)
for doc, meta in zip(mis_resultados['documents'][0], mis_resultados['metadatas'][0]):
    print(f'  {meta["tipo"]} ({meta["barrio"]}): {doc[:80]}...')


mis_resultados_mejorado = collection_mejorada.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

print(f'Consulta: {MI_CONSULTA}')
print("="*10, "Modelo Multilingue", "="*10)
print(f'Consulta: {MI_CONSULTA}')
for doc, meta in zip(mis_resultados_mejorado['documents'][0], mis_resultados_mejorado['metadatas'][0]):
    print(f'  {meta["tipo"]} ({meta["barrio"]}): {doc[:80]}...')


Consulta: Un lugar romantico para ir con mi pareja
========== Modelo BASE ==========
  parrilla (Villa Urquiza): Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las ...
  bar (Palermo): Bar de tragos en Palermo re copado. Buenos cócteles, música en vivo los fines de...
  italiana (San Telmo): Fui a cenar pasta y la verdad que espectacular. Los ñoquis con salsa fileto incr...
Consulta: Un lugar romantico para ir con mi pareja
========== Modelo Multilingue ==========
Consulta: Un lugar romantico para ir con mi pareja
  italiana (San Telmo): Fui a cenar pasta y la verdad que espectacular. Los ñoquis con salsa fileto incr...
  gourmet (Palermo): Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re e...
  cafeteria (Colegiales): Cafetería de especialidad, café re rico, tienen opciones veganas. Ambiente tranq...


In [44]:
MI_CONSULTA = 'Algo para comer rapido al mediodia'

mis_resultados = collection.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

print(f'Consulta: {MI_CONSULTA}')
print("="*10, "Modelo BASE", "="*10)
for doc, meta in zip(mis_resultados['documents'][0], mis_resultados['metadatas'][0]):
    print(f'  {meta["tipo"]} ({meta["barrio"]}): {doc[:80]}...')


mis_resultados_mejorado = collection_mejorada.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

print(f'Consulta: {MI_CONSULTA}')
print("="*10, "Modelo Multilingue", "="*10)
print(f'Consulta: {MI_CONSULTA}')
for doc, meta in zip(mis_resultados_mejorado['documents'][0], mis_resultados_mejorado['metadatas'][0]):
    print(f'  {meta["tipo"]} ({meta["barrio"]}): {doc[:80]}...')

Consulta: Algo para comer rapido al mediodia
========== Modelo BASE ==========
  sushi (delivery): Sushi delivery que pedimos seguido. Fresco, bien armado, llega rápido. No es el ...
  gourmet (Palermo): Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re e...
  bar (Palermo): Bar de tragos en Palermo re copado. Buenos cócteles, música en vivo los fines de...
Consulta: Algo para comer rapido al mediodia
========== Modelo Multilingue ==========
Consulta: Algo para comer rapido al mediodia
  sushi (delivery): Sushi delivery que pedimos seguido. Fresco, bien armado, llega rápido. No es el ...
  hamburguesas (Belgrano): Hamburguesería gourmet. Las papas con cheddar y bacon son adictivas. Burgers de ...
  casera (Boedo): Cantina familiar muy casera. La comida es simple pero rica, como la que hace tu ...


In [45]:
MI_CONSULTA = 'Comida como la que hace la abuela'

mis_resultados = collection.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

print(f'Consulta: {MI_CONSULTA}')
print("="*10, "Modelo BASE", "="*10)
for doc, meta in zip(mis_resultados['documents'][0], mis_resultados['metadatas'][0]):
    print(f'  {meta["tipo"]} ({meta["barrio"]}): {doc[:80]}...')


mis_resultados_mejorado = collection_mejorada.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

print(f'Consulta: {MI_CONSULTA}')
print("="*10, "Modelo Multilingue", "="*10)
print(f'Consulta: {MI_CONSULTA}')
for doc, meta in zip(mis_resultados_mejorado['documents'][0], mis_resultados_mejorado['metadatas'][0]):
    print(f'  {meta["tipo"]} ({meta["barrio"]}): {doc[:80]}...')


Consulta: Comida como la que hace la abuela
========== Modelo BASE ==========
  parrilla (Villa Urquiza): Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las ...
  casera (Boedo): Cantina familiar muy casera. La comida es simple pero rica, como la que hace tu ...
  sushi (delivery): Sushi delivery que pedimos seguido. Fresco, bien armado, llega rápido. No es el ...
Consulta: Comida como la que hace la abuela
========== Modelo Multilingue ==========
Consulta: Comida como la que hace la abuela
  casera (Boedo): Cantina familiar muy casera. La comida es simple pero rica, como la que hace tu ...
  parrilla (Villa Urquiza): Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las ...
  española (Montserrat): Bodegón español tradicional. Las croquetas y la tortilla de papa son de otro pla...


> **✎ Mentor reversible (bonus).** Hasta acá fuiste quien aprende. Ahora dalo vuelta: explicale en treinta segundos, en voz alta o por escrito, la diferencia entre búsqueda semántica y búsqueda por palabras exactas a alguien que nunca escuchó hablar de vectores. Usá el ejemplo del asado si te ayuda. Si esa persona imaginaria lo entiende, es porque vos ya lo entendiste de verdad.

## Vista previa: de la búsqueda a RAG

Lo que hiciste hasta acá es la mitad de RAG: la parte de **Retrieval** (recuperación).

| Paso | Qué hace | Estado |
|---|---|---|
| Guardar documentos con embeddings | ChromaDB.add() | ✅ hecho |
| Buscar los más relevantes | ChromaDB.query() | ✅ hecho |
| Pasarlos como contexto al LLM | prompt con contexto | próximo notebook |
| Generar respuesta fundamentada | Gemini | próximo notebook |

## Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **ChromaDB** | Base vectorial embebida, sin servidor externo |
| **Colección** | Grupo de documentos con embeddings, metadatos e IDs |
| **Búsqueda semántica** | Encuentra por significado, no por palabras exactas |
| **Multilingual-e5** | Modelo de embeddings optimizado para español regional |
| **CRUD vectorial** | add / get / update / delete / query |

**Próximo bloque →** RAG completo: vas a conectar ChromaDB con Gemini y construir un sistema que responde preguntas sobre tus propios documentos.